# DS004940 — stepwise N400 EEG and audio-alignment analysis

This notebook examines the locally downloaded OpenNeuro DS004940 EEG-BIDS dataset without changing any source BDF files. The study contains 136-channel, 512-Hz recordings from the N400 Active and Passive tasks, with target-word timing and the original sentence WAVs. It is a QC and alignment notebook for later EEG-to-audio work, not a completed preprocessing pipeline.

## Execution plan

1. Audit the BIDS tree and identify incomplete BDF-transfer remnants.
2. Index BDF segments and event tables while preserving explicit split-run records.
3. Build one experimental-trial table using `NPC` (congruent) and `NPI` (incongruent) events, anchored at the final target-word onset.
4. Plot trial balance, timing, a representative target-word window, PSD, and condition-wise ERPs.
5. Audit the one-to-one link from experimental events to the public stimulus WAVs.

Run cells in order. The raw data are read lazily, only one representative recording is loaded for QC, and the notebook does not run ICA, reject channels, or overwrite raw data.


In [ ]:
from pathlib import Path
import os
import re
import warnings

os.environ.setdefault('NUMBA_DISABLE_JIT', '1')

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import signal

mne.set_log_level('WARNING')
pd.set_option('display.max_columns', 40)
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 180, 'axes.spines.top': False, 'axes.spines.right': False})

def locate_bundle_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'data' / 'ds004940').exists():
            return candidate
    fallback = Path('/Users/samxie/Research/EEG-Voice/ref_github/speech_decoding/eeg2wave_server_bundle/eeg-recon-0809')
    if (fallback / 'data' / 'ds004940').exists():
        return fallback
    raise FileNotFoundError('Could not find eeg-recon-0809/data/ds004940. Set BUNDLE_ROOT manually.')

BUNDLE_ROOT = locate_bundle_root()
DATA_ROOT = BUNDLE_ROOT / 'data' / 'ds004940'
FIG_DIR = BUNDLE_ROOT / 'reports' / 'generated' / 'ds004940'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f'Bundle root: {BUNDLE_ROOT}')
print(f'Data root:   {DATA_ROOT}')
print(f'Figures:     {FIG_DIR}')


## 1. Inventory and transfer audit

`aws s3 sync` may leave incomplete BDF transfer remnants with a suffix such as `.bdf.<random>`. The cell reports them but never removes them. Files matching the formal `*_eeg.bdf` suffix are the analysis inputs.


In [ ]:
temporary_bdf_files = sorted(DATA_ROOT.rglob('*.bdf.*'))
if temporary_bdf_files:
    print('WARNING: possible incomplete transfer remnants (never analyse these):')
    for path in temporary_bdf_files[:20]:
        print(' -', path.relative_to(DATA_ROOT), f'({path.stat().st_size / 2**20:.1f} MiB)')
else:
    print('No BDF transfer remnants detected.')

for filename in ['dataset_description.json', 'participants.tsv', 'participants.json', 'task-N400Active_eeg.json', 'task-N400Passive_eeg.json']:
    print(f"{'OK' if (DATA_ROOT / filename).exists() else 'MISSING'}  {filename}")
print('Stimulus WAVs:', len(list((DATA_ROOT / 'stimuli').glob('*.wav'))))


## 2. Build a recording-segment index

DS004940 has a few split runs (for example, `sub-004 ... N400Active_run-1/run-2`). The index keeps every BDF segment as a separate row but associates it with the task-level events TSV. Trial construction is intentionally performed once per subject/task below so split runs cannot duplicate trials.


In [ ]:
def parse_bdf_entities(path):
    subject = re.search(r'sub-(\d+)', path.name).group(1)
    task_match = re.search(r'_task-([^_]+)', path.name)
    run_match = re.search(r'_run-(\d+)', path.name)
    return subject, task_match.group(1), run_match.group(1) if run_match else 'single'

def events_for_bdf(bdf_path):
    stem = re.sub(r'_run-\d+(?=_eeg\.bdf$)', '', bdf_path.name)
    return bdf_path.with_name(stem.replace('_eeg.bdf', '_events.tsv'))

rows = []
for bdf in sorted(DATA_ROOT.glob('sub-*/eeg/*_eeg.bdf')):
    subject, task, run = parse_bdf_entities(bdf)
    events = events_for_bdf(bdf)
    channels = bdf.with_name(re.sub(r'_run-\d+(?=_eeg\.bdf$)', '', bdf.name).replace('_eeg.bdf', '_channels.tsv'))
    event_df = pd.read_csv(events, sep='\t') if events.exists() else pd.DataFrame()
    experimental = event_df.get('trial_type', pd.Series(dtype=str)).isin(['NPC', 'NPI'])
    rows.append({
        'subject': subject, 'task': task, 'run': run, 'bdf': bdf, 'events': events, 'channels': channels,
        'bdf_gib': bdf.stat().st_size / 2**30, 'event_table_present': events.exists(),
        'n_event_rows': len(event_df), 'n_experimental_rows_in_event_table': int(experimental.sum()),
    })

recordings = pd.DataFrame(rows).sort_values(['subject', 'task', 'run']).reset_index(drop=True)
assert not recordings.empty, 'No BDF files found under data/ds004940.'
display(recordings[['subject', 'task', 'run', 'bdf_gib', 'event_table_present', 'n_event_rows', 'n_experimental_rows_in_event_table']])
print(f'BDF segments: {len(recordings)} | Subjects: {recordings.subject.nunique()} | BDF volume: {recordings.bdf_gib.sum():.1f} GiB')


## 3. Build the N400 trial and audio index

The experimental conditions are `NPC` (non-prosodic congruent) and `NPI` (non-prosodic incongruent). `onset` is the onset of the final target word and is the ERP anchor; `stim_onset_s_` is retained to calculate sentence-to-target latency. Active-task instruction, prompt, feedback, and response rows are excluded. The filename in `stim_file` is joined to `stimuli/` and validated.


In [ ]:
def build_trial_index(recording_table):
    rows = []
    # events are task-level; take one representative BDF path only to avoid split-run duplication
    task_records = recording_table.sort_values(['subject', 'task', 'run']).drop_duplicates(['subject', 'task'])
    for record in task_records.itertuples(index=False):
        events = pd.read_csv(record.events, sep='\t')
        trials = events.loc[events['trial_type'].isin(['NPC', 'NPI'])].copy().reset_index(names='event_row')
        trials['onset'] = pd.to_numeric(trials['onset'], errors='coerce')
        trials['duration'] = pd.to_numeric(trials['duration'], errors='coerce')
        trials['stim_onset_s_'] = pd.to_numeric(trials['stim_onset_s_'], errors='coerce')
        trials['stim_dur_s_'] = pd.to_numeric(trials['stim_dur_s_'], errors='coerce')
        trials = trials.dropna(subset=['onset']).copy()
        trials['subject'] = record.subject
        trials['task'] = record.task
        trials['trial_in_task'] = np.arange(len(trials))
        trials['condition'] = trials['trial_type'].map({'NPC': 'congruent', 'NPI': 'incongruent'})
        trials['target_onset_s'] = trials['onset']
        trials['sentence_to_target_s'] = trials['onset'] - trials['stim_onset_s_']
        trials['audio_path'] = trials['stim_file'].map(lambda name: DATA_ROOT / 'stimuli' / str(name))
        trials['audio_present'] = trials['audio_path'].map(Path.exists)
        trials['representative_bdf'] = str(record.bdf)
        rows.append(trials)
    return pd.concat(rows, ignore_index=True)

trial_index = build_trial_index(recordings)
trial_index.to_csv(FIG_DIR / 'n400_trial_audio_index.csv', index=False)
display(trial_index[['subject', 'task', 'trial_in_task', 'trial_type', 'condition', 'target_onset_s', 'duration', 'stim_file', 'audio_present']].head(12))
print(f'Experimental trials: {len(trial_index):,} | WAV mappings present: {trial_index.audio_present.mean():.1%}')
assert trial_index.audio_present.all(), 'Some event rows reference a stimulus WAV that is missing locally.'


## 4. Dataset-level trial balance and timing

The left panel shows the `NPC`/`NPI` balance separately for Active and Passive tasks. The right panel shows the distribution of the duration from sentence onset to its final target word, a useful upper bound for a stimulus-to-word reconstruction window.


In [ ]:
condition_counts = pd.crosstab([trial_index['task'], trial_index['condition']], trial_index['subject']).sum(axis=1).unstack(fill_value=0)
latencies = trial_index.dropna(subset=['sentence_to_target_s'])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4), constrained_layout=True)
condition_counts.plot.bar(ax=axes[0], color=['#4C78A8', '#E45756'])
axes[0].set(xlabel='Task / condition', ylabel='Experimental trials', title='N400 trial balance')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Condition', frameon=False)

for (task, condition), group in latencies.groupby(['task', 'condition']):
    axes[1].hist(group['sentence_to_target_s'], bins=30, alpha=0.55, label=f'{task}: {condition}')
axes[1].set(xlabel='Sentence onset → target word onset (s)', ylabel='Trials', title='Final-word timing')
axes[1].legend(frameon=False, fontsize=8)

fig.savefig(FIG_DIR / 'dataset_trial_inventory.png', bbox_inches='tight')
plt.show()


## 5. Select a representative BDF and inspect raw data

The default is an unsplit Passive recording (`sub-001`), selected to make the event table and raw BDF one-to-one. `read_raw_bdf(..., preload=False)` does not place the full recording into memory. Change the subject/task below if required; for split-run tasks, inspect a specified run only after checking which event onsets belong to it.


In [ ]:
EXAMPLE_SUBJECT = '001'
EXAMPLE_TASK = 'N400Passive'
EXAMPLE_TRIAL = 0  # index among NPC/NPI trials within this subject/task

example_record = recordings.query("subject == @EXAMPLE_SUBJECT and task == @EXAMPLE_TASK and run == 'single'").iloc[0]
raw = mne.io.read_raw_bdf(example_record.bdf, preload=False, verbose=False)
example_trials = trial_index.query('subject == @EXAMPLE_SUBJECT and task == @EXAMPLE_TASK').reset_index(drop=True)
example_trial = example_trials.iloc[EXAMPLE_TRIAL]

print(f'BDF: {example_record.bdf.name}')
print(f'Sampling rate: {raw.info["sfreq"]:.0f} Hz | Channels: {len(raw.ch_names)} | Duration: {raw.times[-1] / 60:.1f} min')
display(example_trial[['trial_in_task', 'trial_type', 'condition', 'target_onset_s', 'duration', 'stim_file', 'sentence_to_target_s']].to_frame().T)
print('First channels:', raw.ch_names[:12])


## 6. Target-word window for visual QC

The target-word `onset` is time zero. The bands denote conventional analysis windows: baseline (−0.2–0 s), early auditory (0–0.15 s), N400 candidate (0.25–0.55 s), and later processing (0.55–0.8 s). They are analysis conventions, not measured acoustic boundaries. The plot applies a 1–30 Hz display filter only to the short copied window.


In [ ]:
def available_channels(raw_obj, preferred=('Fz', 'FCz', 'Cz', 'CPz', 'Pz', 'Oz')):
    lookup = {name.lower(): name for name in raw_obj.ch_names}
    return [lookup[name.lower()] for name in preferred if name.lower() in lookup]

def plot_target_word_window(raw_obj, trial, output_path, channels=None, pre_s=0.2, post_s=0.8):
    channels = channels or available_channels(raw_obj)
    if len(channels) < 3:
        channels = raw_obj.ch_names[:min(6, len(raw_obj.ch_names))]
    onset = float(trial['target_onset_s'])
    start, stop = max(0, onset - pre_s), min(float(raw_obj.times[-1]), onset + post_s)
    view = raw_obj.copy().pick(channels).crop(tmin=start, tmax=stop).load_data()
    view.filter(l_freq=1.0, h_freq=30.0, method='iir', verbose=False)
    values_uv = view.get_data() * 1e6
    rel_time = view.times + start - onset
    values_uv -= values_uv[:, rel_time < 0].mean(axis=1, keepdims=True)
    offset = max(20.0, np.nanmedian(np.ptp(values_uv, axis=1)) * 1.25)

    fig = plt.figure(figsize=(15, 8.2), constrained_layout=True)
    grid = fig.add_gridspec(2, 1, height_ratios=[1, 3])
    ax_timeline = fig.add_subplot(grid[0])
    ax_eeg = fig.add_subplot(grid[1], sharex=ax_timeline)
    phases = [(-pre_s, 0.0, '#7D96AC', 'baseline'), (0.0, 0.15, '#F6C85F', 'early\nauditory'),
              (0.15, 0.55, '#9BB8AC', 'N400\ncandidate'), (0.55, post_s, '#EF8A8A', 'late')]
    for phase_start, phase_stop, color, label in phases:
        ax_timeline.axvspan(phase_start, phase_stop, color=color, alpha=0.95)
        ax_eeg.axvspan(phase_start, phase_stop, color=color, alpha=0.13)
        ax_timeline.text((phase_start + phase_stop) / 2, 0.5, label, ha='center', va='center', fontsize=11)
    ax_timeline.set(ylim=(0, 1), yticks=[])
    ax_timeline.tick_params(labelbottom=False)
    for index, (channel, series) in enumerate(zip(channels, values_uv)):
        ax_eeg.plot(rel_time, series + index * offset, lw=0.85, label=channel)
    ax_eeg.axvline(0, color='black', lw=1.1, ls='--', label='target-word onset')
    text = (f"subject={trial['subject']}  task={trial['task']}  trial={int(trial['trial_in_task'])}\n"
            f"condition={trial['condition']}  stimulus={trial['stim_file']}")
    ax_eeg.text(0.01, 0.98, text, transform=ax_eeg.transAxes, va='top', ha='left', bbox={'boxstyle': 'round', 'facecolor': 'white', 'alpha': 0.9})
    ax_eeg.set(xlabel='Time relative to final target-word onset (s)', ylabel='EEG amplitude (µV, offset)')
    ax_eeg.legend(loc='upper right', ncol=2, frameon=True)
    fig.suptitle('DS004940 target-word-locked example trial', fontsize=15)
    fig.savefig(output_path, bbox_inches='tight')
    return fig

trial_figure_path = FIG_DIR / f"target_word_window_sub-{EXAMPLE_SUBJECT}_{EXAMPLE_TASK}_trial-{EXAMPLE_TRIAL:03d}.png"
fig = plot_target_word_window(raw, example_trial, trial_figure_path)
print(f'Saved: {trial_figure_path}')
plt.show()


## 7. PSD and condition-wise target-word ERPs

This is lightweight QC only: up to 100 trials per condition from one recording and a small set of midline channels. It exposes line-noise dominance, broad amplitude anomalies, and gross condition-wise target-word-locked structure before the full preprocessing plan (filtering, bad-channel handling, ocular-artifact treatment, re-referencing, and trial rejection) is chosen.


In [ ]:
PLOT_CHANNELS = available_channels(raw)
MAX_EPOCHS_PER_CONDITION = 100
TMIN, TMAX = -0.2, 0.8

def extract_epochs(raw_obj, onsets, channels, tmin, tmax, max_epochs):
    sfreq = float(raw_obj.info['sfreq'])
    n_before, n_after = int(round(-tmin * sfreq)), int(round(tmax * sfreq))
    kept = []
    for onset in np.asarray(onsets, dtype=float)[:max_epochs]:
        center = int(raw_obj.time_as_index(onset)[0])
        start, stop = center - n_before, center + n_after
        if start < 0 or stop > raw_obj.n_times:
            continue
        values = raw_obj.get_data(picks=channels, start=start, stop=stop) * 1e6
        values -= values[:, :n_before].mean(axis=1, keepdims=True)
        kept.append(values)
    if not kept:
        raise RuntimeError('No valid target-word epochs were found in the selected recording.')
    return np.stack(kept), np.arange(-n_before, n_after) / sfreq

erp = {}
for condition, table in example_trials.groupby('condition'):
    epochs, epoch_times = extract_epochs(raw, table['target_onset_s'], PLOT_CHANNELS, TMIN, TMAX, MAX_EPOCHS_PER_CONDITION)
    erp[condition] = epochs.mean(axis=0)

qc_segment = raw.copy().pick(PLOT_CHANNELS).crop(tmin=float(example_trial['target_onset_s']), tmax=float(example_trial['target_onset_s']) + 20).load_data()
freqs, psd = signal.welch(qc_segment.get_data() * 1e6, fs=raw.info['sfreq'], nperseg=min(4096, qc_segment.n_times), axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.6), constrained_layout=True)
for condition, traces in erp.items():
    axes[0].plot(epoch_times, traces.mean(axis=0), lw=1.5, label=condition)
axes[0].axvline(0, color='black', ls='--', lw=1)
axes[0].axvspan(0.25, 0.55, color='#9BB8AC', alpha=0.25)
axes[0].set(xlabel='Time from target-word onset (s)', ylabel='Midline-channel mean (µV)', title='Condition-wise target-word ERP')
axes[0].legend(frameon=False)
for channel, spectrum in zip(PLOT_CHANNELS, psd):
    axes[1].plot(freqs, 10 * np.log10(spectrum + np.finfo(float).eps), lw=1.1, label=channel)
axes[1].axvline(60, color='gray', ls=':', lw=1)
axes[1].set(xlim=(1, 80), xlabel='Frequency (Hz)', ylabel='Power (dB µV²/Hz)', title='20-second raw-EEG PSD')
axes[1].legend(frameon=False, ncol=2)
fig.savefig(FIG_DIR / 'target_word_locked_qc.png', bbox_inches='tight')
plt.show()


## 8. Public audio audit for EEG-to-audio reconstruction

Unlike DS006104, DS004940 includes its original stimulus WAVs under `stimuli/`. The saved `n400_trial_audio_index.csv` provides the public task/condition/target timing and WAV path for every experimental event. Before model training, determine the reconstruction target deliberately: whole sentence audio, the final word audio segment, or a representation such as a mel spectrogram. Do not treat the BIDS target-word timing as a measured phoneme boundary within the WAV without validating it.


In [ ]:
audio_columns = ['subject', 'task', 'trial_in_task', 'trial_type', 'condition', 'stim_file', 'audio_path', 'audio_present', 'target_onset_s', 'duration']
display(trial_index[audio_columns].head(12))
print('Saved trial/audio manifest:', FIG_DIR / 'n400_trial_audio_index.csv')
print('Unique public WAV files used by experimental trials:', trial_index['stim_file'].nunique())


## Next analysis step

Review the figures and manifest before preprocessing. A defensible next pass should specify, before execution, the channel montage/reference strategy, 60-Hz line-noise treatment, ocular-artifact handling, bad-channel and bad-epoch rules, condition-wise trial counts after QC, and the exact temporal/audio target used for reconstruction. Split BDF runs require an explicit event-to-run boundary check before they enter the same epoching loop.
